# Datamine APTOTRUE Process Example

This notebook demonstrates how to configure and run the **`aptotrue`** process wrapper in `dmstudio`.

## Process Description

## APTOTRUE

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

**APTOTRUE** is designed to be used in conjunction with processes [ANISOANG](<anisoang.md>) and [ESTIMA](<estima.md>) when the apparent dip angle and the true dip direction angle have been interpolated into a model and the true dip angle is required in order to use the Dynamic Anisotropy option to estimate grades. In fact the input file can be any type of file, so long as it contains the apparent dip and true dip direction fields.

The process takes as input the file IN containing fields **APDIP** (apparent dip) and **TRDIPDIR** (true dip direction) and parameter **APDIPDIR** (apparent dip direction) and calculates the **TRDIP** (true dip) field which is written to file **OUT**. Parameter **APDIPDIR** is the azimuth of the sections corresponding to the apparent dip field.

### Input Files:

* **in** (Undefined):
  Input file containing the true dip direction and the apparent dip angle fields.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing all the fields in the input file plus the true dip angle.
  Required=Yes

### Fields:

* **apdip** (Numeric : IN):
  Apparent dip angle. If not specified then APDIP is assumed.
  Default=APDIP
  Required=No

* **trdipdir** (Numeric : IN):
  True dip direction angle. If not specified then TRDIPDIR is assumed.
  Default=TRDIPDIR
  Required=No

* **trdip** (Numeric : OUT):
  True dip angle. If not specified then TRDIP will be created.
  Default=TRDIP
  Required=No

### Parameters:

* **apdipdir**:
  Apparent dip direction angle in degrees. This is the azimuth of the sections on which
  the original string data was digitised.
  Range=0,360
  Values=Undefined
  Default=0
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('aptotrue')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute aptotrue
print("Running aptotrue...")
dm_cmd.aptotrue(
    in_i='t_assays',  # required
    out_o='t_aptotrue_out',  # required
    trdip_f='optional',  # required
    apdipdir_p='required_val',  # required
    # apdip_f='optional',  # optional
    # trdipdir_f='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("aptotrue execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_aptotrue_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")